# Analysis

Sentiment, categorization, clustering, and Plotly exploration for discovery records.

**Prereq:** `python3 setup/extract_records.py` so `output/processed/challenges.csv` and `expectations.csv` exist.


In [13]:
%pip install -q -r requirements.txt
%pip install -q "nbformat>=4.2.0"


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [14]:
from pathlib import Path
import sys

import hdbscan
import numpy as np
import pandas as pd
import plotly.express as px
import umap
from IPython.display import Markdown, display
from sklearn.metrics import silhouette_score

sys.path.insert(0, str(Path(".").resolve()))
from setup.analyze_records import (
    CATEGORY_CONFIG,
    CATEGORY_DESCRIPTIONS,
    realign_by_sentiment,
    run_categorize,
    run_sentiment,
)
from setup.extract_records import SOURCE_MEETING_NOTES, load_prepared_records

CHALLENGES_CSV = Path("./output/processed/challenges.csv")
EXPECTATIONS_CSV = Path("./output/processed/expectations.csv")
PROCESSED_DIR = Path("./output/processed")

CHALLENGES_SCORED = PROCESSED_DIR / "challenges_scored.csv"
EXPECTATIONS_SCORED = PROCESSED_DIR / "expectations_scored.csv"
CATEGORIZED_CHALLENGES_CSV = PROCESSED_DIR / "categorized_challenges.csv"
CATEGORIZED_EXPECTATIONS_CSV = PROCESSED_DIR / "categorized_expectations.csv"
CATEGORY_SUMMARY_CSV = PROCESSED_DIR / "category_summary.csv"

CHALLENGE_TEXT_COL = "pain_points"
EXPECTATION_TEXT_COL = "expectations"


## 1. Load & prep records

Loads CSVs via `load_prepared_records()` (focus-group aliases, source tags, short meeting-note merge).


In [15]:
df_painpoints, df_expectations = load_prepared_records(
    CHALLENGES_CSV, EXPECTATIONS_CSV
)

print(f"Challenges: {len(df_painpoints)} | Expectations: {len(df_expectations)}")
print(df_painpoints["source"].value_counts().to_string())


  Redacted stakeholder names (78 dictionary entries → [PERSON])
Challenges: 1110 | Expectations: 375
source
meeting_notes    914
worksheet        196


## 2. Sentiment

Uses `Thi144/sentiment-distilbert-7class`, mapped to negative / neutral / positive.


In [16]:
df_painpoints, df_expectations = run_sentiment(df_painpoints, df_expectations)

print("Pain sentiment:\n", df_painpoints["sentiment"].value_counts().to_string())
print("\nExpectation sentiment:\n", df_expectations["sentiment"].value_counts().to_string())

df_painpoints.to_csv(CHALLENGES_SCORED, index=False, encoding="utf-8-sig")
df_expectations.to_csv(EXPECTATIONS_SCORED, index=False, encoding="utf-8-sig")
print(f"Saved → {CHALLENGES_SCORED.name}, {EXPECTATIONS_SCORED.name}")


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Scoring challenges…
Scoring expectations…
Pain sentiment:
 sentiment
negative    895
positive    215

Expectation sentiment:
 sentiment
positive    334
negative     41
Saved → challenges_scored.csv, expectations_scored.csv


## 3. Realign misclassified rows

Move positive “pain” → expectations and negative “expectations” → pain, **meeting_notes only**.


In [17]:
n_pos = int(
    ((df_painpoints["source"] == SOURCE_MEETING_NOTES) & (df_painpoints["sentiment"] == "positive")).sum()
)
n_neg = int(
    ((df_expectations["source"] == SOURCE_MEETING_NOTES) & (df_expectations["sentiment"] == "negative")).sum()
)

df_painpoints, df_expectations = realign_by_sentiment(
    df_painpoints, df_expectations, only_source=SOURCE_MEETING_NOTES
)

print(f"Moved meeting_notes only: {n_pos} positive→expectations, {n_neg} negative→pain")
print(f"Shapes: pain {len(df_painpoints)}, expectations {len(df_expectations)}")
print(df_painpoints["source"].value_counts().to_string())


Moved meeting_notes only: 195 positive→expectations, 22 negative→pain
Shapes: pain 937, expectations 548
source
meeting_notes    741
worksheet        196


## 4. Categorize records

Hybrid: title keywords → body keywords → semantic similarity.


In [18]:
df, expectations_df, challenge_embeddings, embedder = run_categorize(
    df_painpoints, df_expectations
)

print("Challenge categories:\n", df["Category"].value_counts().to_string())
print("\nExpectation categories:\n", expectations_df["Category"].value_counts().to_string())


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/15 [00:00<?, ?it/s]

Batches:   0%|          | 0/9 [00:00<?, ?it/s]

Challenge categories:
 Category
Records & Document Management       207
Case Management                     199
System Integration                  126
User Experience & Performance        93
Communication & Collaboration        88
Workflow & Business Processes        76
Scheduling & Resource Management     60
Data Management & Visibility         51
Training & Documentation             21
Reporting & Decision Support         16

Expectation categories:
 Category
Records & Document Management       132
Case Management                     103
System Integration                   82
Communication & Collaboration        54
User Experience & Performance        45
Workflow & Business Processes        40
Scheduling & Resource Management     30
Data Management & Visibility         29
Reporting & Decision Support         18
Training & Documentation             15


## 5. Cluster challenges (UMAP + HDBSCAN)


In [26]:
def assign_cluster_names(
    frame: pd.DataFrame,
    cluster_col: str = "Cluster",
    category_col: str = "Category",
    output_col: str = "Cluster_Label",
) -> tuple[pd.DataFrame, pd.DataFrame]:
    cluster_summary = (
        frame.groupby([cluster_col, category_col]).size().reset_index(name="Count")
    )
    dominant = (
        cluster_summary.sort_values("Count", ascending=False)
        .drop_duplicates(subset=[cluster_col])
        [[cluster_col, category_col]]
        .rename(columns={category_col: output_col})
    )
    return frame.merge(dominant, on=cluster_col, how="left"), cluster_summary


def soft_assign_noise(
    coords: np.ndarray,
    labels: np.ndarray,
    radius_percentile: float = 92,
    radius_slack: float = 1.35,
) -> tuple[np.ndarray, int]:
    """Pull leftover noise into the nearest cluster if it sits inside that cluster's envelope."""
    out = labels.copy()
    clustered = out != -1
    if not clustered.any():
        return out, 0

    centroids, radii = {}, {}
    for cid in sorted(set(out[clustered])):
        pts = coords[out == cid]
        centroids[cid] = pts.mean(axis=0)
        dists = np.linalg.norm(pts - centroids[cid], axis=1)
        radii[cid] = float(np.percentile(dists, radius_percentile) * radius_slack)

    assigned = 0
    for i in np.where(out == -1)[0]:
        best_c, best_d = None, np.inf
        for cid, center in centroids.items():
            d = float(np.linalg.norm(coords[i] - center))
            if d < best_d:
                best_c, best_d = cid, d
        if best_c is not None and best_d <= radii[best_c]:
            out[i] = best_c
            assigned += 1
    return out, assigned


TARGET_CLUSTERS = len(CATEGORY_CONFIG)
# Prefer coverage over ultra-tight cores (prior sweep left ~36% as noise).
NOISE_PENALTY = 1.35
CLUSTER_COUNT_PENALTY = 0.012

reducer = umap.UMAP(
    n_neighbors=15, n_components=12, min_dist=0.0, metric="cosine", random_state=42
)
reduced = reducer.fit_transform(challenge_embeddings)

candidate_params = sorted(
    {
        (mcs, ms, method)
        for mcs in range(5, 14)
        for ms in (1, 2, 3, max(1, mcs // 3))
        for method in ("eom", "leaf")
    }
)

sweep_rows, best_labels, best_params, best_score = [], None, None, -np.inf
n_rows = max(len(df), 1)
for mcs, ms, method in candidate_params:
    labels = hdbscan.HDBSCAN(
        min_cluster_size=mcs,
        min_samples=ms,
        metric="euclidean",
        cluster_selection_method=method,
    ).fit_predict(reduced)

    mask = labels != -1
    n_clusters = len(set(labels[mask]))
    noise = int((labels == -1).sum())
    noise_frac = noise / n_rows
    silhouette = (
        silhouette_score(reduced[mask], labels[mask], metric="euclidean")
        if mask.sum() > 1 and n_clusters > 1
        else -1.0
    )
    # Reject near-all-noise or absurd fragmentation early.
    if noise_frac > 0.45 or n_clusters < 4 or n_clusters > 40:
        combined = -1.0
    else:
        combined = (
            silhouette
            - abs(n_clusters - TARGET_CLUSTERS) * CLUSTER_COUNT_PENALTY
            - noise_frac * NOISE_PENALTY
        )
    sweep_rows.append({
        "min_cluster_size": mcs,
        "min_samples": ms,
        "method": method,
        "clusters": n_clusters,
        "noise": noise,
        "noise_pct": round(noise_frac * 100, 1),
        "silhouette": round(float(silhouette), 4),
        "combined_score": round(float(combined), 4),
    })
    if combined > best_score:
        best_score, best_labels, best_params = combined, labels, (mcs, ms, method)

print("Best HDBSCAN params:", best_params, "score:", round(float(best_score), 4))
display(pd.DataFrame(sweep_rows).sort_values("combined_score", ascending=False).head(10))

raw_labels = best_labels.copy()
raw_noise = int((raw_labels == -1).sum())
best_labels, n_soft = soft_assign_noise(reduced, best_labels)
print(
    f"Soft-assigned {n_soft} noise points into nearest clusters "
    f"(noise {raw_noise} → {int((best_labels == -1).sum())})"
)

df["Cluster_Raw"] = raw_labels
df["Cluster"] = best_labels
df["Assign_Status"] = np.where(
    raw_labels != -1,
    "HDBSCAN cluster",
    np.where(best_labels != -1, "Soft-assigned", "Unclustered"),
)
df, cluster_summary = assign_cluster_names(df)
df.loc[df["Cluster"] == -1, "Cluster_Label"] = "Noise / unclustered"

print("\nAssignment status:")
print(df["Assign_Status"].value_counts().to_string())
print("\nCluster → label counts:")
print(df.groupby(["Cluster", "Cluster_Label"]).size().to_string())
print(
    f"\nUnclustered: {(df['Cluster'] == -1).sum()} / {len(df)} "
    f"({100 * (df['Cluster'] == -1).mean():.1f}%)"
)


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Best HDBSCAN params: (13, 4, 'eom') score: 0.0506


,min_cluster_size,min_samples,method,clusters,noise,noise_pct,silhouette,combined_score
56,13,4,eom,19,193,20.6,0.4367,0.0506
42,12,1,eom,24,160,17.1,0.4395,0.0409
50,13,1,eom,23,172,18.4,0.4404,0.0366
48,12,4,eom,21,234,25.0,0.4942,0.0251
38,11,2,eom,25,173,18.5,0.4535,0.0242
44,12,2,eom,24,184,19.6,0.4545,0.0214
52,13,2,eom,23,196,20.9,0.4551,0.0167
51,13,1,leaf,24,205,21.9,0.4800,0.0166
26,9,2,eom,27,164,17.5,0.4492,0.0089
32,10,2,eom,27,164,17.5,0.4492,0.0089


Soft-assigned 64 noise points into nearest clusters (noise 193 → 129)

Assignment status:
Assign_Status
HDBSCAN cluster    744
Unclustered        129
Soft-assigned       64

Cluster → label counts:
Cluster  Cluster_Label      
-1       Noise / unclustered    129

Unclustered: 129 / 937 (13.8%)


### Soft assignment map

2D projection of the same UMAP space used for HDBSCAN. Color by assignment status to see which points were dense cores, which were rescued by soft assignment, and which stayed unclustered.


### Cluster contents

Browse every cluster for dissection, including noise (`Cluster == -1`).


In [29]:
cols = [
    c
    for c in [
        "Cluster",
        "Cluster_Label",
        "Assign_Status",
        "focus_group",
        CHALLENGE_TEXT_COL,
        "Category",
        "source",
    ]
    if c in df.columns
]
show = df[cols].sort_values(["Cluster", "Assign_Status", "focus_group"]).reset_index(drop=True)

print("Cluster sizes (including unclustered):")
display(
    show.groupby(["Cluster", "Cluster_Label"], as_index=False)
    .size()
    .rename(columns={"size": "Count"})
    .sort_values("Cluster")
)

for cid, group in show.groupby("Cluster", sort=True):
    label = group["Cluster_Label"].iloc[0]
    display(Markdown(f"### Cluster {cid} — {label} ({len(group)} records)"))
    display(
        group[[c for c in ["focus_group", CHALLENGE_TEXT_COL, "Category", "source"] if c in group.columns]]
        .reset_index(drop=True)
    )


Cluster sizes (including unclustered):


,Cluster,Cluster_Label,Count
0,-1,Noise / unclustered,129


### Cluster -1 — Noise / unclustered (129 records)

,focus_group,pain_points,Category,source
0,Admin Aide,Sometimes Rent Holds / Rent Release doesn't ap...,Scheduling & Resource Management,worksheet
1,Admin Aide,They could edit it on their end,Communication & Collaboration,meeting_notes
2,Admin Aide,His work around – said Law verify address,Records & Document Management,meeting_notes
3,Admin Aide,Spell check or grammar check,Records & Document Management,meeting_notes
4,Admin Aide,Issue with feature – put in a complaint under ...,Case Management,meeting_notes
...,...,...,...,...
124,Zoning,Lot aleration,Workflow & Business Processes,worksheet
125,Zoning,Basement filling system (excel spreadsheet),Workflow & Business Processes,worksheet
126,Zoning,"Communication: applicant, reviewer",Communication & Collaboration,worksheet
127,Zoning,Biggest weakness with inspectors – proactive e...,Workflow & Business Processes,meeting_notes


### Cluster 0 — nan (15 records)

...

,focus_group,pain_points,Category,source
0,Admin Aide,Would save time if all info on catchment map w...,Records & Document Management,meeting_notes
1,Admin Aide,Can they have assessment update the address?,Records & Document Management,meeting_notes
2,Admin Aide,Assessment change of address form – share,Records & Document Management,meeting_notes
3,Admin Aide,Main place where parcel is,Records & Document Management,meeting_notes
4,BAA Supervisors,Too many interfaces not linking info (i.e. par...,Records & Document Management,worksheet
5,BAA Supervisors,Address search less reliable – ideally would b...,Records & Document Management,meeting_notes
6,BAA Supervisors,Which apartment of the structure is cited? Dis...,Records & Document Management,meeting_notes
7,BAA Supervisors,Image Mate Online – assessed owner addresses,Case Management,meeting_notes
8,CPC,Property addresses same in eTax – shared asses...,Records & Document Management,meeting_notes
9,CPO-Coordinators,Parcel search can be tricky,Records & Document Management,meeting_notes


### Cluster 17 — nan (16 records)

,focus_group,pain_points,Category,source
0,Admin Aide,With LLCs do have to keep it distinct if multi...,Data Management & Visibility,meeting_notes
1,Admin Aide,Can't find new owners – creates an issue becau...,Communication & Collaboration,meeting_notes
2,BAA ALJs,One owner with multiple properties,Case Management,worksheet
3,BAA ALJs,One property to multiple owner,Case Management,worksheet
4,BAA ALJs,"When ownership changes, prior owner's fines/fe...",Workflow & Business Processes,meeting_notes
5,BAA ALJs,"Multiple owners and same owner re-cited, doubl...",Case Management,meeting_notes
6,BAA ALJs,"When PO owns multiple properties, no way to vi...",Case Management,meeting_notes
7,CPC,If unpaid and sold to new owner – new owner no...,Case Management,meeting_notes
8,CPC,Could pay it and sue civilly – technically new...,Case Management,meeting_notes
9,CPO-Coordinators,Updating owner information,Records & Document Management,worksheet


### Cluster 18 — nan (36 records)

,focus_group,pain_points,Category,source
0,Admin Aide,Cannot search person with associated addresses...,Records & Document Management,worksheet
1,Admin Aide,Reverts to former address updated addresses do...,Records & Document Management,worksheet
2,Admin Aide,M: Pixel perfect when searching by address in ...,Records & Document Management,meeting_notes
3,Admin Aide,Updating address only annoying,Records & Document Management,meeting_notes
4,Admin Aide,Sheer amount of duplicates in address book in ...,System Integration,meeting_notes
5,Admin Aide,Put in contact information – enter someone wit...,Data Management & Visibility,meeting_notes
6,Admin Aide,Automatically adds hypens with phone number,Communication & Collaboration,meeting_notes
7,Admin Aide,Can see contact phone numbers,Communication & Collaboration,meeting_notes
8,Admin Aide,That needs to be updated,Training & Documentation,meeting_notes
9,BAA Supervisors,Unreliable address search,Records & Document Management,worksheet


In [23]:
# add a new column called cluster_number
df["cluster_number"] = df["Cluster"]

# add a new column called cluster_label
df["cluster_label"] = df["Cluster_Label"]


## 6. Visualizations


In [ ]:
category_counts = (
    df.groupby("Category", as_index=False)
    .size()
    .rename(columns={"size": "Count"})
    .sort_values("Count", ascending=True)
)
fig_cat = px.bar(
    category_counts, x="Count", y="Category", orientation="h",
    title="Pain point categories", color="Count", color_continuous_scale="Blues",
)
fig_cat.update_layout(showlegend=False, height=500, template="plotly_white")
fig_cat.show()

expectation_counts = (
    expectations_df.groupby("Category", as_index=False)
    .size()
    .rename(columns={"size": "Count"})
    .sort_values("Count", ascending=True)
)
fig_exp_cat = px.bar(
    expectation_counts, x="Count", y="Category", orientation="h",
    title="Expectation categories", color="Count", color_continuous_scale="Teal",
)
fig_exp_cat.update_layout(showlegend=False, height=500, template="plotly_white")
fig_exp_cat.show()


In [35]:
coords_2d = umap.UMAP(
    n_neighbors=15, n_components=2, min_dist=0.1, metric="cosine", random_state=42
).fit_transform(challenge_embeddings)
plot_df = pd.DataFrame({
    "x": coords_2d[:, 0], "y": coords_2d[:, 1],
    "Category": df["Category"].values,
    "Cluster_Label": df["Cluster_Label"].values,
    "Challenge": df[CHALLENGE_TEXT_COL].values,
    "Focus Group": df["focus_group"].values,
})
fig_map = px.scatter(
    plot_df, x="x", y="y", color="Category",
    hover_data=["Focus Group", "Cluster_Label", "Challenge"],
    title="2D semantic map by category",
)
fig_map.update_layout(height=560, width=1100, template="plotly_white")
fig_map.show()

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [33]:
ch_c = df["Category"].value_counts()
ex_c = expectations_df["Category"].value_counts()
cats = sorted(set(ch_c.index) | set(ex_c.index), key=lambda c: -(ch_c.get(c, 0) + ex_c.get(c, 0)))
plot_cmp = pd.DataFrame({
    "Category": cats,
    "Challenges": [int(ch_c.get(c, 0)) for c in cats],
    "Expectations": [int(ex_c.get(c, 0)) for c in cats],
}).melt(id_vars="Category", var_name="Type", value_name="Count")
fig_cmp = px.bar(
    plot_cmp, x="Count", y="Category", color="Type", barmode="group", orientation="h",
    title="Challenges vs expectations by theme",
    color_discrete_map={"Challenges": "#E76F51", "Expectations": "#2A9D8F"},
)
fig_cmp.update_layout(height=520, template="plotly_white", yaxis=dict(categoryorder="total ascending"))
fig_cmp.show()

In [ ]:
top_n = 18
top = df.groupby("focus_group").size().sort_values(ascending=False).head(top_n)
exp = expectations_df.groupby("focus_group").size()
plot_top = pd.DataFrame({
    "Focus Group": top.index.tolist(),
    "Challenges": top.values.astype(int),
    "Expectations": [int(exp.get(g, 0)) for g in top.index],
}).melt(id_vars="Focus Group", var_name="Kind", value_name="Count")
fig_top = px.bar(
    plot_top,
    x="Focus Group",
    y="Count",
    color="Kind",
    barmode="group",
    title=f"Top {top_n} focus groups",
    color_discrete_map={"Challenges": "#E76F51", "Expectations": "#2A9D8F"},
    category_orders={"Focus Group": top.index.tolist()},
)
fig_top.update_layout(
    height=560,
    width=1100,
    template="plotly_white",
    xaxis=dict(tickangle=-40),
    margin=dict(b=140),
)
fig_top.show()


In [31]:
heatmap_df = pd.crosstab(df["focus_group"], df["Category"])
focus_group_order = heatmap_df.sum(axis=1).sort_values(ascending=False).index
category_order = heatmap_df.sum(axis=0).sort_values(ascending=False).index
heatmap_df = heatmap_df.loc[focus_group_order, category_order].T
fig_heatmap = px.imshow(
    heatmap_df,
    labels=dict(x="Focus Group", y="Category", color="Count"),
    # title="Focus group vs category",
    color_continuous_scale="YlOrRd", text_auto=True, aspect="auto",
)
fig_heatmap.update_layout(height=700, width=1100, xaxis_tickangle=-45, template="plotly_white")
fig_heatmap.show()

## 7. Save categorized outputs


In [25]:
challenge_cols = [
    c for c in [
        "department", "focus_group", CHALLENGE_TEXT_COL, "processed_text", "source", "sentiment",
        "title", "Category", "Category_Method", "Category_Confidence",
        "Cluster", "Cluster_Label",
    ] if c in df.columns
]
expectation_cols = [
    c for c in [
        "department", "focus_group", EXPECTATION_TEXT_COL, "processed_text", "source", "sentiment",
        "title", "Category", "Category_Method", "Category_Confidence",
    ] if c in expectations_df.columns
]

df[challenge_cols].to_csv(CATEGORIZED_CHALLENGES_CSV, index=False, encoding="utf-8-sig")
expectations_df[expectation_cols].to_csv(
    CATEGORIZED_EXPECTATIONS_CSV, index=False, encoding="utf-8-sig"
)
category_summary = (
    df.groupby("Category", as_index=False).size()
    .rename(columns={"size": "Count"})
    .sort_values("Count", ascending=False)
)
category_summary.to_csv(CATEGORY_SUMMARY_CSV, index=False, encoding="utf-8-sig")

print(f"Saved {CATEGORIZED_CHALLENGES_CSV} ({len(df)} rows)")
print(f"Saved {CATEGORIZED_EXPECTATIONS_CSV} ({len(expectations_df)} rows)")
print(f"Saved {CATEGORY_SUMMARY_CSV}")
display(category_summary)


Saved output/processed/categorized_challenges.csv (937 rows)
Saved output/processed/categorized_expectations.csv (548 rows)
Saved output/processed/category_summary.csv


,Category,Count
3,Records & Document Management,207
0,Case Management,199
6,System Integration,126
8,User Experience & Performance,93
1,Communication & Collaboration,88
9,Workflow & Business Processes,76
5,Scheduling & Resource Management,60
2,Data Management & Visibility,51
7,Training & Documentation,21
4,Reporting & Decision Support,16
